In [ ]:
import pandas as pd

In [ ]:
df_input2 = pd.read_excel(file_path, sheet_name="Hoja2")
df_input2

In [ ]:
words2 = [x.split("|") for x in df_input2["words"]][0]
words2 = [x.replace(",", "").replace("A1","").replace("A2","").replace("B1","").replace("B2","").replace("./pro/","").replace("prep.","").replace("conj.",
         "").replace("/pro","").replace("det.","") for x in words2]
words2

In [ ]:
pd.DataFrame({"word":words2}).to_excel("oxford.xlsx", index=False)

In [ ]:


# ---------------------------
# 1. Leer Excel (columna A)
# ---------------------------
file_path = "G:\\Mi unidad\\Python\\english\\words.xlsx"  # <-- tu archivo

df_input = pd.read_excel(file_path, sheet_name="Hoja1")

# Asumimos que la columna A es la primera
words = df_input.iloc[:, 0].dropna().str.lower().unique()
words = list(words)
print(len(words))
print(words)

In [ ]:
# 1. Leer Excel (columna A)
# ---------------------------
file_path = "G:\\Mi unidad\\Python\\english\\phrases.xlsx"  # <-- tu archivo

df_input = pd.read_excel(file_path, sheet_name="Hoja1")

# Asumimos que la columna A es la primera
words = df_input.iloc[:, 0].dropna().str.lower().unique()
words = list(words)
print(len(words))
print(words)

In [ ]:
import pandas as pd
from deep_translator import GoogleTranslator
import time


translator = GoogleTranslator(source='en', target='es')

translations = []

for i, word in enumerate(words):
    try:
        translated = translator.translate(word)
    except:
        translated = "ERROR"
    
    ax = pd.DataFrame({
        "word": [word],
        "translation": [translated]
    })
    translations.append(ax)
    

    print(f"{i+1}/{len(words)}: {word} - {translated}")

translations = pd.concat(translations)

In [ ]:
translations

In [ ]:
import numpy as np

translations["freq"] = np.round(np.random.uniform(3, 6.3, size=len(translations)), decimals=3)
translations

In [ ]:
df= pd.read_excel("traducciones.xlsx")
df
df = pd.concat([df, translations], ignore_index=True)
df = df.sort_values(by="freq", ascending=False).reset_index(drop=True).drop_duplicates(subset="word", keep="first")
df["translation"] = df["translation"].str.replace(".","").str.replace("¿","").str.replace("?","")
df.to_excel("traducciones.xlsx", index=False)
df.head(20)


In [ ]:
df["freq"].describe()

In [ ]:
from wordfreq import zipf_frequency

df["freq"] = df["word"].apply(lambda x: zipf_frequency(x, 'en'))

df = df.sort_values(by="freq", ascending=False).reset_index(drop=True)

# ---------------------------
df = pd.DataFrame(df)
df.to_excel("traducciones.xlsx", index=False)
df

df

In [ ]:
import pandas as pd

del df["freq"]

df_flashcards = pd.DataFrame({
    "front": df["word"],
    "back": df["translation"]
})

df_flashcards.to_csv("flashcards.csv", index=False)
df_flashcards

In [ ]:
# import time
# for i in range(0, len(df)):
#     print(f"{df.loc[i, 'word']}: {df.loc[i, 'translation']}")
#     time.sleep(3) 

In [8]:
import gradio as gr
import pandas as pd
import unicodedata
import os
import random
from gtts import gTTS

# ---------------------------
# Cargar datos
# ---------------------------
df = pd.read_excel("traducciones.xlsx")

for col in ["correct_count", "wrong_count", "is_difficult", "needs_second_pass"]:
    if col not in df.columns:
        df[col] = 0

df = df.sort_values(by="freq", ascending=False).reset_index(drop=True)

# ---------------------------
# Preview DF
# ---------------------------
def get_df_preview(n=300):
    ax = df[df["is_difficult"] == 1]
    return ax.head(n)

# ---------------------------
# Constantes
# ---------------------------
COOLDOWN_CORRECT = 14
COOLDOWN_WRONG = 8
COOLDOWN_SECOND_PASS = 25

# ---------------------------
# Métricas
# ---------------------------
def get_metrics(df):
    mastered  = (df["correct_count"] >= 3).sum()
    learning  = (df["correct_count"] < 3).sum()
    difficult = (df["is_difficult"] == 1).sum()
    return mastered, learning, difficult

# ---------------------------
# Audio
# ---------------------------
def generate_audio(word):
    file = "audio.mp3"
    gTTS(text=word, lang='en').save(file)
    return file

# ---------------------------
# Estado
# ---------------------------
def init_state():
    return {
        "cooldown": {},
        "word_counter": 0,
        "current": None,
    }

# ---------------------------
# Selección de palabra
# ---------------------------
def get_next_word(state, only_difficult):
    counter = state["word_counter"]
    candidates = []

    for _, row in df.iterrows():
        word = row["word"]

        if row["correct_count"] >= 3:
            continue

        if only_difficult and row["is_difficult"] != 1:
            continue

        if counter < state["cooldown"].get(word, 0):
            continue

        priority = row["freq"] + (row["wrong_count"] * 50)
        candidates.append((priority, row))

    if not candidates:
        state["current"] = None
        return state

    candidates.sort(key=lambda x: x[0], reverse=True)
    chosen = candidates[0][1]

    state["current"] = chosen.to_dict()
    state["word_counter"] += 1
    return state

# ---------------------------
# UI helpers
# ---------------------------
def format_word(word):
    return f"<div style='font-size:45px; text-align:center; font-weight:bold;'>{word}</div>"

def show_translation(state):
    if state["current"] is None:
        return ""
    return f"<div style='font-size:28px; text-align:center;'>📖 {state['current']['translation']}</div>"

# ---------------------------
# RESPUESTAS
# ---------------------------
def answer_known(state, only_difficult):
    if state["current"] is None:
        return "⏳ Sin palabras", None, "", "", "", "", "", state, get_df_preview()

    word = state["current"]["word"]
    idx = df.index[df["word"] == word][0]
    counter = state["word_counter"]

    df.loc[idx, "correct_count"] += 1
    df.loc[idx, "wrong_count"] = 0

    if df.loc[idx, "needs_second_pass"] == 1:
        df.loc[idx, "correct_count"] = 3
    else:
        df.loc[idx, "needs_second_pass"] = 1
        state["cooldown"][word] = counter + COOLDOWN_SECOND_PASS

    state["cooldown"][word] = counter + COOLDOWN_CORRECT
    df.to_excel("traducciones.xlsx", index=False)

    state = get_next_word(state, only_difficult)

    if state["current"] is None:
        return "⏳ Fin", None, "", "", "", "", "", state, get_df_preview()

    next_word = state["current"]["word"]
    audio = generate_audio(next_word)
    m, l, d = get_metrics(df)

    return format_word(next_word), audio, "", "", str(m), str(l), str(d), state, get_df_preview()

def answer_skip(state, only_difficult):
    if state["current"] is None:
        return "⏳ Sin palabras", None, "", "", "", "", "", state, get_df_preview()

    word = state["current"]["word"]
    translation = state["current"]["translation"]
    idx = df.index[df["word"] == word][0]
    counter = state["word_counter"]

    df.loc[idx, "wrong_count"] += 1
    state["cooldown"][word] = counter + COOLDOWN_WRONG
    df.to_excel("traducciones.xlsx", index=False)

    translation_html = f"<div style='font-size:28px; text-align:center;'>❌ {translation}</div>"

    state = get_next_word(state, only_difficult)

    if state["current"] is None:
        return "⏳ Fin", None, "", translation_html, "", "", "", state, get_df_preview()

    next_word = state["current"]["word"]
    audio = generate_audio(next_word)
    m, l, d = get_metrics(df)

    return format_word(next_word), audio, "", translation_html, str(m), str(l), str(d), state, get_df_preview()

def mark_difficult(state, only_difficult):
    if state["current"] is None:
        return "⏳ Sin palabra", None, "", "", "", "", "", state, get_df_preview()

    word = state["current"]["word"]
    idx = df.index[df["word"] == word][0]

    df.loc[idx, "is_difficult"] = 1
    state["cooldown"][word] = 100
    df.loc[idx, "correct_count"] = 3
    df.to_excel("traducciones.xlsx", index=False)

    state = get_next_word(state, only_difficult)

    if state["current"] is None:
        return "⏳ Fin", None, "", "", "", "", "", state, get_df_preview()

    next_word = state["current"]["word"]
    audio = generate_audio(next_word)
    m, l, d = get_metrics(df)

    return format_word(next_word), audio, "", "", str(m), str(l), str(d), state, get_df_preview()

def mark_mastered(state, only_difficult):
    if state["current"] is None:
        return "⏳ Sin palabra", None, "", "", "", "", "", state, get_df_preview()

    word = state["current"]["word"]
    idx = df.index[df["word"] == word][0]

    df.loc[idx, "correct_count"] = 3
    df.to_excel("traducciones.xlsx", index=False)

    state = get_next_word(state, only_difficult)

    if state["current"] is None:
        return "⏳ Fin", None, "", "", "", "", "", state, get_df_preview()

    next_word = state["current"]["word"]
    audio = generate_audio(next_word)
    m, l, d = get_metrics(df)

    return format_word(next_word), audio, "", "", str(m), str(l), str(d), state, get_df_preview()

# ---------------------------
# START
# ---------------------------
def start_app(state, only_difficult):
    state = get_next_word(state, only_difficult)

    if state["current"] is None:
        return "⏳ Sin palabras", None, "", "", "", "", "", state, get_df_preview()

    word = state["current"]["word"]
    audio = generate_audio(word)
    m, l, d = get_metrics(df)

    return format_word(word), audio, "", "", str(m), str(l), str(d), state, get_df_preview()

# ---------------------------
# UI
# ---------------------------
with gr.Blocks(css="""
#btn_1 { background-color:#A2D668 !important; color:white; border-radius:10px; }
#btn_2 { background-color:#F77A1D !important; color:white; border-radius:10px; }
#btn_3 { background-color:#7030A0 !important; color:white; border-radius:10px; }
#btn_4 { background-color:#46B1E1 !important; color:white; border-radius:10px; }
#btn_5 {
    background-color: #FF6B6B !important;
    color: white !important;
    border-radius: 10px;
}
""") as app:

    state = gr.State(init_state())

    gr.Markdown("<h1 style='text-align:center;'>Train My English</h1>")

    only_difficult = gr.Checkbox(label="😈 Solo palabras difíciles")

    word = gr.Markdown()
    feedback = gr.Markdown()

    show_btn = gr.Button("Ver traducción", elem_id="btn_4")

    with gr.Row():
        known_btn = gr.Button("Conocida", elem_id="btn_1")
        skip_btn = gr.Button("Pasar", elem_id="btn_2")
        master_btn = gr.Button("Dominada", elem_id="btn_3")
        difficult_btn = gr.Button("Difícil", elem_id="btn_5")

    audio = gr.Audio(autoplay=True)

    mastered = gr.Textbox(label="🏆 Dominadas")
    learning = gr.Textbox(label="📚 Aprendiendo")
    difficult = gr.Textbox(label="😈 Difíciles")

    dummy1 = gr.Textbox(visible=False)

    # 👇 DataFrame visual limitado
    df_view = gr.Dataframe(label="📊 Dataset", row_count=500, interactive=True)

    outputs = [word, audio, dummy1, feedback, mastered, learning, difficult, state, df_view]

    show_btn.click(show_translation, inputs=[state], outputs=feedback)
    known_btn.click(answer_known, inputs=[state, only_difficult], outputs=outputs)
    skip_btn.click(answer_skip, inputs=[state, only_difficult], outputs=outputs)
    master_btn.click(mark_mastered, inputs=[state, only_difficult], outputs=outputs)
    difficult_btn.click(mark_difficult, inputs=[state, only_difficult], outputs=outputs)

    app.load(start_app, inputs=[state, only_difficult], outputs=outputs)

app.launch()

C:\Users\PC\AppData\Local\Temp\ipykernel_3240\3407892290.py:226: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: css. Please pass these parameters to launch() instead.
  with gr.Blocks(css="""


* Running on local URL:  http://127.0.0.1:7866
* To create a public link, set `share=True` in `launch()`.


In [ ]:
# Nuevo
import gradio as gr
import pandas as pd, numpy as np
import os
import random
from gtts import gTTS


ef = 2.5
cooldown_default = 10.0 # cd por defecto al inicio para todas las palabras

df = pd.read_excel("traducciones.xlsx")
df["score"] = df["score"].astype(float)
df["cooldown"] = cooldown_default

for col in []:
    if col not in df.columns:
        df[col] = 0

# df = df.sort_values(by="freq", ascending=False).reset_index(drop=True)
df["hour"] = pd.to_datetime(df["hour"])
df["elapsed"] = 99999

df.head(10)


In [ ]:
import pandas as pd
from datetime import datetime

# Candidatos
temp = df.loc[(df["is_dominated"] == 0) & (df["elapsed"] > df["cooldown"])]

if temp.empty:
    raise ValueError("No hay palabras disponibles")

idx = temp.index[0]
first = df.at[idx, "word"]
print("First word:", first)

# Respuesta del usuario
q = 5
# pasar: 0 
# mala: 1 
# dificil = 3 
# correcta: 4 
# dominada: 5

old_ef = df.at[idx, "score"]
new_ef = old_ef + (0.1 - (5 - q) * (0.08 + (5 - q) * 0.02))

cd = df.at[idx, "cooldown"]
new_cd = cd * new_ef

now = datetime.now().replace(microsecond=0)
df.at[idx, "hour"] = now

df["elapsed"] = (now - df["hour"]).dt.total_seconds().fillna(99999)


# Actualizar contadores
if q == 0:
    df.at[idx, "num_pass"] += 1

elif q in [1, 3]:
    df.at[idx, "num_bad"] += 1
    if q == 3:
        df.at[idx, "num_hard"] += 1

elif q in [4, 5]:
    df.at[idx, "num_correct"] += 1

    if q == 5: 
        is_dominated = True 
        df.at[idx, "is_dominated"] = 1 
        count_ini = df.at[idx, "num_dominated"]
        df.at[idx, "num_dominated"] += 1 
        count_new = df.at[idx, "num_dominated"]

        print(count_ini, count_new) 
        if count_new > count_ini: 
            print("count_new > count_ini") 
            df.at[idx, "cooldown"] = 33

df.loc[df["num_dominated"] >= 2, "exit"] = 1
df.loc[df["elapsed"] > df["cooldown"], "cooldown"] = cooldown_default
df.loc[df["elapsed"] > df["cooldown"], "elapsed"] = 99999

# Actualizar score
df.at[idx, "score"] = new_ef

if df.at[idx, "is_dominated"] == 0:
    df.at[idx, "cooldown"] = new_cd

print("new ef:", new_ef)
print("new_cd:", new_cd)
df.head(20)